<div style="border:1px solid #d9e1ea;border-left:6px solid #2d8a57;border-radius:14px;padding:18px 20px;background:#fff;"><h1 style="margin:0 0 6px;color:#16213b;">LangChain — Complete Learning Notebook</h1><p style="margin:0;color:#63738a;">Models, structured output, prompts, runnables, memory, chains, RAG, evaluation, agents and tools — mapped to the Reservation Analytics AI Agent.</p><p style="margin:10px 0 0;"><code>notebooks/langchain/langchain_project_learning.ipynb</code></p></div>

### How to use this notebook

Read the **mental model → small code example → project mapping → Q&A** for each topic.

This goes broader than the current project. The current project specifically uses `langchain_openai.ChatOpenAI(...).with_structured_output(...)` for extraction, while **LlamaIndex** owns knowledge RAG and **LangGraph** owns workflow orchestration.

![LangChain ecosystem](assets/01_ecosystem.svg)

## 1. What LangChain Is

**LangChain** is a framework for building LLM-powered applications by connecting language models with prompts, tools, structured output, memory, and external systems.

**LangGraph**. It manages the stateful workflow, routing, validation, clarification, and transitions between nodes.

**LangChain handles LLM interaction; LangGraph controls the workflow.**

In this project, LangChain is not the orchestration framework. 
LangGraph handles the workflow, while LangChain is used selectively for LLM integration and structured extraction.

| Aspect          | LangChain                         | LangGraph                                |
| --------------- | --------------------------------- | ---------------------------------------- |
| Main role       | LLM integration                   | Workflow orchestration                   |
| Best for        | Prompts, tools, structured output | State, routing, multi-step flow          |
| In this project | Structured extraction             | Main agent workflow                      |
| Simple example  | Question → Pydantic output        | Extract → Validate → Resolve → Analytics |


**Current project positioning**

```text
Reservation Project
├── LangChain / langchain_openai → structured extraction
├── LangGraph                  → state + routing
├── LlamaIndex                 → RAG
├── FAISS                      → vector search
├── Pydantic                   → typed schemas
└── FastAPI                    → service API
```

![Learning path](assets/02_learning_path.svg)

In [5]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

/Users/B1/ghome/github/online/reservation-analytics-ai-agent


## 2. Models and ChatOpenAI

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

## 3. Prompts

In [3]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract business context. Never invent missing values."),
    ("human", "{question}"),
])

## 4. Structured Output — used by this project

![Project mapping](assets/03_project_mapping.svg)

In [9]:
from langchain_openai import ChatOpenAI
from app.analytics.models.request import ExtractedRequest
from app.settings import Settings

settings = Settings()

llm = ChatOpenAI(
    model=settings.openai_model,
    api_key=settings.openai_api_key,
    temperature=0,
).with_structured_output(ExtractedRequest)

question = "How many reserved users for Phone Mi 17 Pro in Germany?"

request = llm.invoke(
    "Extract intent, metric, detail request, country, product and campaign context. "
    "Never invent missing values.\n\nQuestion: " + question
)
request

ExtractedRequest(intent='analytics', metric='number of reserved users', detail_requested=False, query=ReservationQuery(country='Germany', country_code=None, product='Phone Mi 17 Pro', campaign_id=None, campaign_name=None, campaign_month=None, campaign_year=None))

## 5. Runnable and LCEL

Modern LangChain components commonly implement the **Runnable** interface: `invoke()`, `batch()`, `stream()` and pipe composition with `|`.

Mental model: `Prompt | Model | Parser`

In [ ]:
chain = prompt | llm
result = chain.invoke({"question": "What is the reservation conversion rate?"})
result

## 6. Memory

Memory retains useful context across turns. Distinguish **LangChain memory concepts**, **LangGraph State**, and **LangGraph checkpointing**. The current Reservation project does not need LangChain memory in its main analytics flow.

## 7. Chains

A chain is a deterministic composition of steps. Chains fit fixed paths; LangGraph is stronger when you need branching, clarification, retries or pause/resume.

## 8. Document Q&A / RAG

LangChain can build RAG too, but this project intentionally uses **LlamaIndex** as the higher-level knowledge layer.

## 9. Evaluation

Evaluate extraction accuracy, routing, retrieval relevance, grounded answers, campaign resolution, metric correctness and clarification behavior with repeatable datasets.

## 10. Agents and Tools

A tool is an external capability such as search, calculator, database lookup or internal API. This project stays deterministic: LangGraph chooses routes and Python invokes services; the LLM does not receive unrestricted database-tool access.

## 11. LangChain vs LangGraph vs LlamaIndex

| Technology | Main responsibility |
|---|---|
| **LangChain / ChatOpenAI** | Structured extraction |
| **LangGraph** | State + workflow |
| **LlamaIndex** | Knowledge RAG |
| **FAISS** | Vector search |
| **Pydantic** | Typed contracts |
| **FastAPI** | HTTP serving |

## Q&A — Fast Review

<details open><summary><b>Q1. Does this project use LangChain?</b></summary>

**Answer:** Yes. `app/graph/nodes/extract.py` uses `langchain_openai.ChatOpenAI` with structured output.

</details>

<details open><summary><b>Q2. Chain vs Agent?</b></summary>

**Answer:** A chain follows a mostly predefined sequence; an agent chooses actions dynamically.

</details>

<details open><summary><b>Q3. Why not use LangChain for RAG too?</b></summary>

**Answer:** It could, but LlamaIndex keeps the knowledge layer focused and explicit.

</details>

<details open><summary><b>Q4. What is LCEL?</b></summary>

**Answer:** LangChain Expression Language is pipe-style Runnable composition.

</details>

<details open><summary><b>Q5. How to harden extraction?</b></summary>

**Answer:** Typed schemas, no-guess prompts, validation, regression datasets and tracing.

</details>

## Classic Architecture Q&A — Memorize This

### Q. Why do you use LangChain selectively instead of making it the entire application framework?

> **I use LangChain selectively rather than making it the entire application framework. LangChain's ChatOpenAI integration handles structured extraction, LangGraph handles stateful workflow orchestration, and LlamaIndex with FAISS handles the knowledge RAG layer. This keeps responsibilities explicit and prevents the LLM from directly controlling analytics SQL.**

<div style="background:#eef7ff;border:1px solid #c9e0f2;border-radius:10px;padding:10px 12px;margin:10px 0;">
</div>

### Memory Map

```text
LangChain / ChatOpenAI  → Structured Extraction
Pydantic                → Typed Contract
LangGraph               → Stateful Workflow
LlamaIndex + FAISS      → Knowledge RAG
Controlled SQL          → Trusted Numbers
FastAPI                 → Service API
```

### One-line takeaway

> **Do not force every responsibility into one framework. Keep the boundaries explicit.**